# 5. Interpretabilidade e ensaio da escoragem

Antes da escolha final do modelo devemos analisar:

1. **Interpretabilidade:** como e quanto cada variável afeta o score do modelo
2. **Ensaio da escoragem:** a função que vai escorar a OOT cega é testada numa amostra da base de desenvolvimento, pelo mesmo caminho (CSV bruto, tratamento, modelo, arquivo de saída)

Interpretação SHAP: a contribuição de uma variável está na escala de log-odds de adimplência. Positivo aumenta o score (menos risco); negativo diminui. Para cada cliente, valor base + soma das contribuições = log-odds do score. As contribuições são exatas (TreeSHAP do próprio LightGBM). As VPs não têm significado conhecido, então a leitura é "quanto e em que direção", sem história causal. Isso evita o erro da apresentação de 2025, que leu a importância por ganho como se fosse efeito.

## 5.1 Imports, leitura e o modelo

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from equifax.config import MODELS, OOT_PATH, REPORTS, TABLES, RANDOM_SEED
from equifax.conjuntos import carregar_conjunto
from equifax.dados import carregar_desenvolvimento, janelas
from equifax.escoragem import (colunas_brutas_do_desenvolvimento, ensaiar_com_desenvolvimento,
                               escorar_arquivo, salvar_pacote)
from equifax.experimentos import escorar, treinar, treinar_final
from equifax.explicar import (conferir_soma, contribuicoes, escolher_perfis, grafico_beeswarm,
                              grafico_dependencia, grafico_importancia, grafico_shap_por_safra,
                              grafico_waterfall, importancia_shap, shap_por_safra)
from equifax.viz import salvar_figura

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
PARAMS_M1 = json.load(open(REPORTS / "params_m1.json", encoding="utf-8"))
VARIAVEIS = carregar_conjunto(PARAMS_M1["conjunto"])
AMOSTRA_SHAP = 10000  # linhas do teste usadas no SHAP

base = carregar_desenvolvimento()
partes = janelas(base)
treino, teste = partes["treino"], partes["teste"]

lgbm = treinar("lgbm", PARAMS_M1["lgbm"], treino, VARIAVEIS, semente=RANDOM_SEED)
logit = treinar("logit_woe", PARAMS_M1["logit_woe"], treino, VARIAVEIS)
print("Modelos treinados no treino:", PARAMS_M1["conjunto"], len(VARIAVEIS), "variáveis")

## 5.2 Contribuições SHAP do LightGBM

Calculadas numa amostra do teste. A soma das contribuições com o valor base precisa reproduzir o score de cada cliente: é a conferência de que o cálculo está certo. No beeswarm, a cor indica o valor da variável (vermelho = alto); para as categóricas (`estado`, faixas), a cor é só o código da categoria e não tem ordem.

In [ ]:
contrib, valor_base, x = contribuicoes(lgbm, teste, amostra=AMOSTRA_SHAP, semente=RANDOM_SEED)
safra_amostra = teste.loc[contrib.index, "safra"]
ok, erro = conferir_soma(lgbm, contrib, valor_base, x)
print(f"Soma confere: {ok} (maior erro {erro:.2e}); valor base = {valor_base:.3f} log-odds")

imp = importancia_shap(contrib)
imp.to_frame("media_abs_shap").to_csv(TABLES / "shap_m1_lgbm_importancia.csv")
print("Variáveis com contribuição média acima de 0,01:", int((imp > 0.01).sum()), "de", len(imp))
imp.head(20).round(4)

In [ ]:
grafico_importancia(contrib, n=20, titulo="Importância SHAP no teste (M1, LightGBM)")
salvar_figura("m1_shap_importancia")
plt.show()

grafico_beeswarm(contrib, valor_base, x, n=15, titulo="Direção e dispersão do efeito (M1, LightGBM)")
salvar_figura("m1_shap_beeswarm")
plt.show()

## 5.3 Dependência das 5 principais

Como a contribuição muda com o valor da variável. Os códigos especiais (-1, -2, 9999) aparecem em vermelho, à esquerda, com o efeito médio escrito.

In [ ]:
for var in imp.index[:5]:
    grafico_dependencia(contrib, x, var)
    salvar_figura(f"m1_shap_dependencia_{var}")
    plt.show()

## 5.4 Por que um cliente recebeu aquele score

Três perfis do teste: um de score baixo (5% de menor score, risco alto), um mediano e um de score alto (5% de maior score). O gráfico parte do valor base e soma a contribuição de cada variável até chegar ao score do cliente.

In [ ]:
score_amostra = escorar(lgbm, teste.loc[contrib.index])
perfis = escolher_perfis(score_amostra, indice=contrib.index)
descricao = {"q05": "score baixo (risco alto)", "q50": "score mediano", "q95": "score alto (risco baixo)"}
for nome, indice in perfis.items():
    grafico_waterfall(contrib, valor_base, x, indice, n=12, titulo=f"Perfil {nome}: {descricao[nome]}")
    salvar_figura(f"m1_shap_waterfall_{nome}")
    plt.show()

## 5.5 A explicação é estável no tempo?

Média do |SHAP| das 10 principais variáveis em cada safra do teste. Linhas planas indicam que o modelo usa as variáveis do mesmo jeito em todos os meses. A tabela mostra o coeficiente de variação entre as safras.

In [ ]:
top10 = list(imp.index[:10])
media, media_abs = shap_por_safra(contrib, safra_amostra, top10)
grafico_shap_por_safra(media_abs, "Média do |SHAP| por safra no teste (10 principais, M1)")
salvar_figura("m1_shap_por_safra")
plt.show()
(media_abs.std() / media_abs.mean()).rename("coeficiente_de_variacao").round(3).to_frame()

### Comparação com a logística com WoE

Se os dois modelos, construídos de formas muito diferentes, apontam as mesmas variáveis, a explicação fica mais confiável. Na logística, a contribuição é exata: coeficiente × WoE.

In [ ]:
contrib_logit, base_logit, x_logit = contribuicoes(logit, teste.loc[contrib.index])
imp_logit = importancia_shap(contrib_logit)
comuns = set(imp.index[:20]) & set(imp_logit.index[:20])
print(f"{len(comuns)} das 20 principais variáveis são as mesmas nos dois modelos")
pd.DataFrame({"posicao_lgbm": imp.rank(ascending=False).astype(int),
              "posicao_logistica": imp_logit.rank(ascending=False, method="min").astype(int)}).loc[imp.index[:20]]

## 5.6 Ensaio da escoragem com a base de desenvolvimento

O modelo é salvo num pacote (em `models/`, fora do Git). Uma amostra do CSV bruto de desenvolvimento, sem o target, é escorada pelo mesmo caminho que a OOT vai seguir. O score precisa ser idêntico ao calculado direto na base tratada, e o arquivo de saída precisa ser o arquivo de entrada com uma coluna a mais.

In [ ]:
pacote = salvar_pacote(lgbm, MODELS / "m1_lgbm_ensaio.joblib", treino, colunas_brutas_do_desenvolvimento(),
                       nome="M1 LightGBM (ensaio, treinado só no treino)")
resumo = ensaiar_com_desenvolvimento(pacote, base, n=5000)
resumo

## 5.7 Escoragem da OOT cega (Passo 10: NÃO rodar antes)

Fica pronta, mas desligada. Só roda depois da escolha do vencedor (Passo 8), da decisão sobre o retreino (8 ou 10 safras) e da resposta da organização sobre o formato do score.

In [ ]:
ESCORAR_OOT = False  # só no Passo 10

if ESCORAR_OOT:
    JANELA = ("202301", "202308")  # ou ("202301", "202310"), decisão em aberto
    vencedor = treinar_final("lgbm", PARAMS_M1["lgbm"], base, VARIAVEIS, janela=JANELA)
    usado = base[base["safra"].astype(int).between(int(JANELA[0]), int(JANELA[1]))]
    pacote_final = salvar_pacote(vencedor, MODELS / "vencedor.joblib", usado,
                                 colunas_brutas_do_desenvolvimento(), nome="vencedor")
    saida, relatorio = escorar_arquivo(OOT_PATH, pacote_final, n_linhas_esperado=None,  # número de linhas informado pela organização
                                       nome_coluna="score", tipo_score="probabilidade")
    print(json.dumps(relatorio, indent=2, ensure_ascii=False, default=str))
else:
    print("Escoragem da OOT desligada (Passo 10).")

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.